In [1]:
pip install pyspark==3.0.1

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import requests

In [2]:
spark = SparkSession.builder \
    .appName("JupyterKafkaRealTime") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.0.1"
    ) \
    .getOrCreate()

In [3]:
spark.sparkContext.setLogLevel("WARN")

In [4]:
user_schema = StructType([
    StructField("location", StructType([
        StructField("country", StringType())
    ])),
    StructField("dob", StructType([
        StructField("age", IntegerType())
    ]))
])

In [5]:
kafka_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "user-events") \
    .load()

In [6]:
json_parsed_df = kafka_stream_df.selectExpr("CAST(value AS STRING) as json_data") \
    .select(from_json(col("json_data"), user_schema).alias("user")) \
    .select("user.location.country", "user.dob.age")

In [7]:
country_counts_df = json_parsed_df.groupBy("country").count()

In [8]:
def send_batch_to_influx(df, batch_id):
    records = df.collect()
    for row in records:
        if row['country']:
            clean_country = row['country'].replace(" ", "\\ ")
            count_val = row['count']
            line_protocol_data = f"user_counts,country={clean_country} count={count_val}"
            try:

                requests.post("http://127.0.0.1:8086/write?db=user_analytics", data=line_protocol_data, timeout=3)
            except:
                pass

In [9]:
query = country_counts_df.writeStream \
    .foreachBatch(send_batch_to_influx) \
    .outputMode("complete") \
    .start()